# Transfer Learning: ResNet18 Feature Extraction & Fine-Tuning
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/04_Deep_Learning/computer-vision/transfer_learning_resnet.ipynb)

Training CNNs from scratch needs millions of images. Transfer learning reuses ImageNet-learned features: freeze the backbone, swap the classifier head, optionally unfreeze later with a tiny LR.

Dataset: CIFAR-10 subset (fast on free Colab GPU).

In [ ]:
!pip install -q torch torchvision

## 1. Data + transforms matching ResNet expectations

In [ ]:
import torch, torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats
tf = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(mean, std)])

train_full = torchvision.datasets.CIFAR10(".", train=True, transform=tf, download=True)
test_full  = torchvision.datasets.CIFAR10(".", train=False, transform=tf, download=True)

train_ds = Subset(train_full, range(8000))     # small on purpose
test_ds  = Subset(test_full, range(2000))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=256, num_workers=2)

## 2. Strategy A - frozen backbone (feature extraction)

In [ ]:
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

def build(freeze_backbone=True):
    m = torchvision.models.resnet18(weights="IMAGENET1K_V1")
    for p in m.parameters():
        p.requires_grad = not freeze_backbone
    m.fc = nn.Linear(m.fc.in_features, 10)      # new head, trainable by default
    return m.to(device)

model = build(freeze_backbone=True)
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # ONLY head params
loss_fn = nn.CrossEntropyLoss()

## 3. Train + evaluate helper

In [ ]:
def run_epoch(dl, train=True):
    model.train() if train else model.eval()
    tot, correct, lsum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            lsum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            tot += len(y)
    return lsum / tot, correct / tot

for epoch in range(3):
    tr_loss, tr_acc = run_epoch(train_dl)
    te_loss, te_acc = run_epoch(test_dl, train=False)
    print(f"epoch {epoch}: train acc={tr_acc:.3f}  test acc={te_acc:.3f}")

## 4. Strategy B - unfreeze layer4 with discriminative LR

In [ ]:
for p in model.layer4.parameters():
    p.requires_grad = True

opt = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])
for epoch in range(2):
    tr_loss, tr_acc = run_epoch(train_dl)
    te_loss, te_acc = run_epoch(test_dl, train=False)
    print(f"finetune epoch {epoch}: test acc={te_acc:.3f}")

## Decision guide
| Data available | Approach |
|---|---|
| tiny (<1k/class) | frozen backbone + linear probe |
| medium | last block unfreezing (above) |
| large & domain far away (medical) | full fine-tune, low LR everywhere |

Sanity checks: input normalized with the SAME stats the backbone trained with; augment train only; monitor train-vs-test gap for silent overfitting.